In [ ]:
# BERT + DeiT III: one bounded, resumable scientific training session; no test.
import base64, hashlib, json, os, shutil, signal, subprocess, sys, time, zlib
from pathlib import Path

STARTED = time.monotonic()
SESSION_SECONDS = 27000
STOP_EPOCH = 4
PREVIOUS_RECEIPT = 'cb845905ff96584520e8776cb81ba23437dd78518e0e8776cb81ba23437dd7851'
QUOTA_USED_MINUTES = 0.01
bundle = json.loads(zlib.decompress(base64.b64decode('...')))
source = Path('/tmp/urop_bert_deit3_source')
for name, item in bundle.items():
    target = source/name; target.parent.mkdir(parents=True, exist_ok=True)
    raw = base64.b64decode(item['bytes'])
    if hashlib.sha256(raw).hexdigest() != item['sha256']:
        raise ValueError('Bundled source hash mismatch: '+name)
    target.write_bytes(raw)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(4*1024**2), b''): h.update(block)
    return h.hexdigest()

def unique(pattern, digest):
    matches = [p for p in Path('/kaggle/input').glob(pattern) if sha(p) == digest]
    if len(matches) != 1: raise ValueError(f'Expected exactly one verified {pattern}, found {len(matches)}')
    return matches[0]

manifest = unique('**/urop-faculty-usable-packed-input-v3/data/official_paired_v2/paired_manifest.csv',
                  '96c84e9453401cc4a24999c41cdbeb182dafd08771383cf1c75feea9c3001caf')
data_root = manifest.parents[2]
coverage = unique('**/urop_mm_packed_gate_20261026/packed_runtime_coverage.json',
                  'afe8c723f91fe74f791908e0c16fab94d919af64c7cb7c9b5a37e676361d95e5')
index = coverage.with_name('packed_image_index.sqlite')
if sha(index) != 'fc8dc03a4f773da5762891c8392375add5fb8087251b527f3b37a880308e6018': raise ValueError('Index differs')
text = unique('**/urop-text-only-final-results/**/bert-base/best_weights.pt',
              '870bea50fa608dddd20421c21fd1786e244b1323e22734093214b87023666753')
image = unique('**/urop-image-only-vit-83-training/urop_vit83/full_full_frame/best_epoch3.pt',
               '97cb89e986c2481d65dd61ff4f2570a907898c885c34ea2e9460385ce01ae300')
preflight = unique('**/urop_bert_deit3_preflight/preflight.json',
                   'd4ab864c2ce9208015e5826685668c4eef17d8ec8b45dc941e9e0582ebb9ba96')
receipt = json.loads(preflight.read_text())
if receipt['status'] != 'passed' or receipt['device'] != 'cuda': raise ValueError('Preflight did not pass')
if receipt['estimated_epoch_seconds_excluding_checkpoint'] <= 0: raise ValueError('Bad throughput receipt')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_name(0) != receipt['gpu']:
    raise ValueError('Different or unavailable GPU; repeat preflight')
output = Path('/kaggle/working/urop_bert_deit3_training')
if output.exists(): raise ValueError('Preserve existing output; do not duplicate a run')
if PREVIOUS_RECEIPT:
    previous = unique('**/urop_bert_deit3_training/session_receipt.json', PREVIOUS_RECEIPT)
    old = json.loads(previous.read_text())
    if old['state'] not in ('stopped_at_boundary', 'stage_complete'): raise ValueError('Predecessor not terminal')
    shutil.copytree(previous.parent, output)
    (output/'session_receipt.json').unlink()
    if (output/'trainer.lock').exists(): raise ValueError('Preserve predecessor lock for investigation')
stage = 'part1' if STOP_EPOCH <= 2 else 'part2'
cmd = [sys.executable, '-u', str(source/'model/text_plus_image/bert_deit3/train.py'),
       '--root', str(data_root), '--coverage', str(coverage), '--packed-index', str(index),
       '--config', str(source/'configs/text_plus_image/bert_deit3/experiment.json'),
       '--output', str(output), '--warmstart', str(text), '--image-warmstart', str(image),
       '--preflight', str(preflight), '--stage', stage, '--stop-after-epoch', str(STOP_EPOCH)]
if PREVIOUS_RECEIPT: cmd += ['--resume']
env = {**os.environ, 'CUDA_VISIBLE_DEVICES':'0', 'CUBLAS_WORKSPACE_CONFIG':':4096:8',
        'TOKENIZERS_PARALLELISM':'false', 'OMP_NUM_THREADS':'1', 'MKL_NUM_THREADS':'1'}
print('TRAINING_SESSION_REGISTERED', json.dumps({'quota_used_minutes':QUOTA_USED_MINUTES,
      'session_seconds': SESSION_SECONDS,'stop_after_epoch':STOP_EPOCH,'previous_receipt':PREVIOUS_RECEIPT,
      'preflight_sha256':sha(preflight),'no_test':True}), flush=True)
proc = subprocess.Popen(cmd, env=env)
stop_sent = False
while proc.poll() is None:
    if time.monotonic()-STARTED >= SESSION_SECONDS-1200 and not stop_sent:
        proc.send_signal(signal.SIGTERM); stop_sent = True
        print('STOP_REQUESTED checkpoint boundary', flush=True)
    if time.monotonic()-STARTED >= SESSION_SECONDS-120 and proc.poll() is None:
        proc.kill(); proc.wait(); raise RuntimeError('No clean stop; preserve this run for inspection')
    time.sleep(10)
if proc.returncode != 0: raise RuntimeError(f'Trainer failed with exit code {proc.returncode}')
sys.path.insert(0,str(source/'training/text_plus_image/Scripts'))
from multimodal_recovery_v1 import load_generation
payload, proof = load_generation(output/'recovery')
state = payload['state']
if state['cursor'] != payload['cursor']: raise ValueError('Recovery cursor mismatch')
complete = len(state['history']) >= STOP_EPOCH
summary = {'state':'stage_complete' if complete else 'stopped_at_boundary', 'stop_after_epoch':STOP_EPOCH,
           'history_epochs':len(state['history']), 'phase':state['phase'], 'epoch':state['epoch'],
           'cursor':state['cursor'], 'recovery':proof, 'best':state['best'],
           'protocol_sha256':sha(output/'protocol.json'), 'preflight_sha256':sha(preflight),
           'elapsed_seconds':time.monotonic()-STARTED, 'no_test':True}
tmp = output/'session_receipt.tmp'; tmp.write_text(json.dumps(summary,indent=2)+'\n'); os.replace(tmp,output/'session_receipt.json')
print('TRAINING_SESSION_RECEIPT',json.dumps(summary),flush=True)
print('SESSION_RECEIPT_SHA256',sha(output/'session_receipt.json'),flush=True)
